In [ ]:
pip install --upgrade pip setuptools wheel
!pip install -r requirements.txt --verbose

  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 30.4 MB/s  0:00:00
Using cached setuptools-82.0.0-py3-none-any.whl (1.0 MB)

  Attempting uninstall: wheel

    Found existing installation: wheel 0.45.1

    Uninstalling wheel-0.45.1:

      Successfully uninstalled wheel-0.45.1

  Attempting uninstall: setuptools

    Found existing installation: setuptools 80.9.0

    Uninstalling setuptools-80.9.0:

   ------------- -------------------------- 1/3 [setuptools]
   ------------- -------------------------- 1/3 [setuptools]
   ------------- -------------------------- 1/3 [setuptools]
   ------------- -------------------------- 1/3 [setuptools]
   ------------- -------------------------- 1/3 [setuptools]
      Successfully uninstalled setuptools-80.9.0
   ------------- -------------------------- 1/3 [setuptools]
   ------------- -------------------

In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

#Lecture des données
X_train = pd.read_csv("../data/X_train_update.csv")
y_train = pd.read_csv("../data/Y_train_CVw08PX.csv")
X_test = pd.read_csv("../data/X_test_update.csv")

#Affichage des tailles des datasets
print(f"Taille X_train : {X_train.shape}")
print(f"Taille Y_train : {y_train.shape}")
print(f"Taille X_test : {X_test.shape}")

#Merge données d'entrainement
full_data = pd.merge(X_train, y_train, left_index=True, right_index=True)

#Suppression de la colonne Unnamed: 0_y qui est une colonne d'index inutile
full_data = full_data.drop(['Unnamed: 0_y'], axis=1)

#Renomage de la colonne Unnamed: 0_x en id et mise en index de cette colonne
full_data.rename(columns={'Unnamed: 0_x': 'id'}, inplace=True)
full_data.set_index(['id'], inplace=True)

X_test.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
X_test.set_index(['id'], inplace=True)

Taille X_train : (84916, 5)
Taille Y_train : (84916, 2)
Taille X_test : (13812, 5)


In [ ]:
# Informations sur les données du dataset d'entrainement
print("Types de données :")
print(full_data.dtypes)
print("\nValeurs manquantes :")
print(full_data.isnull().sum())
print("\nPourcentage de valeurs manquantes :")
print((full_data.isnull().sum() / len(full_data) * 100))

# nan_count = (full_data['description'].astype(str) == 'nan').sum()
# print(f"\nNombre de description NaN : {nan_count}")

Types de données :
designation    object
description    object
productid       int64
imageid         int64
prdtypecode     int64
dtype: object

Valeurs manquantes :
designation        0
description    29800
productid          0
imageid            0
prdtypecode        0
dtype: int64

Pourcentage de valeurs manquantes :
designation     0.000000
description    35.093504
productid       0.000000
imageid         0.000000
prdtypecode     0.000000
dtype: float64

Nombre de description NaN : 29800


In [63]:
# Statistiques descriptives
print("\nStatistiques descriptives :")
#Longueur des textes
full_data['len_designation'] = (full_data['designation'].astype(str).apply(len))
full_data['len_description'] = (full_data['description'].astype(str).apply(len))

#Nombre de mots
full_data['nb_words_designation'] = (full_data['designation'].astype(str).apply(lambda x: len(x.split())))
full_data['nb_words_description'] = (full_data['description'].astype(str).apply(lambda x: len(x.split())))

print("\nDESIGNATION")
print(f"Longueur minimale du titre : {full_data['len_designation'].min()} caractères")
print(f"Longueur maximale du titre : {full_data['len_designation'].max()} caractères")
print(f"Longueur moyenne du titre : {full_data['len_designation'].mean():.2f} caractères")
print(f"Longueur médiane du titre : {full_data['len_designation'].median()} caractères")
print(f"Nombre moyen de mots : {full_data['nb_words_designation'].mean():.2f} mots")


print("\nDESCRIPTION")
print(f"Longueur minimale de la description : {full_data['len_description'].min()} caractères")
print(f"Longueur maximale de la description : {full_data['len_description'].max()} caractères")
print(f"Longueur moyenne de la description : {full_data['len_description'].mean():.2f} caractères")
print(f"Longueur médiane de la description : {full_data['len_description'].median()} caractères")
print(f"Nombre moyen de mots : {full_data['nb_words_description'].mean():.2f} mots")


Statistiques descriptives :

DESIGNATION
Longueur minimale du titre : 11 caractères
Longueur maximale du titre : 250 caractères
Longueur moyenne du titre : 70.16 caractères
Longueur médiane du titre : 64.0 caractères
Nombre moyen de mots : 11.56 mots

DESCRIPTION
Longueur minimale de la description : 1 caractères
Longueur maximale de la description : 12451 caractères
Longueur moyenne de la description : 525.61 caractères
Longueur médiane de la description : 231.0 caractères
Nombre moyen de mots : 80.52 mots


In [64]:
# Distribution des catégories
count_cat = full_data['prdtypecode'].value_counts().sort_index()
print(f"Nombre de catégories : {len(count_cat)}")


Nombre de catégories : 27


In [91]:
#Fonction de nettoyage du texte
import re

def clean_text(text):
    if pd.isnull(text):
        return ""

    # Supprimer la ponctuation
    #text = re.sub(r'[^\w\s]', '', text)

    # Supprimer balises HTML
    text = re.sub(r'<.*?>', '', text)

    # Remplacer <br /> par un espace
    text = text.replace(r'<br />', ' ')

    # Remplacer les référence de caractère HTML
    text = text.replace(r'&amp;', '&')
    text = text.replace(r'&nbsp;', ' ')
    text = text.replace(r'&lt', '<')
    text = text.replace(r'&gt', '>')
    text = text.replace(r'&quot', '"')
    text = text.replace(r'&#39', "'")
    text = text.replace(r'&eacute', 'e')
    text = text.replace(r'&egrave', 'e')
    text = text.replace(r'&ecirc', 'e')
    
    # Convertir en minuscules
    text = text.lower()

    # Supprimer les espaces supplémentaires
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [96]:
#Nettoyage du texte
full_data['clean_designation'] = full_data['designation'].apply(clean_text)
full_data['clean_description'] = full_data['description'].apply(clean_text)

#Vérification de doublon dans les désignations et descriptions nettoyées
duplicate_designation = full_data['clean_designation'].duplicated().sum()
duplicate_description = full_data['clean_description'].duplicated().sum()

#Concaténation des désignations et descriptions nettoyées pour l'analyse de texte
full_data['text'] = full_data['clean_designation'] + ' ' + full_data['clean_description']

In [ ]:
print(duplicate_description)
print(duplicate_designation)


37555
2705


In [80]:
full_data.head()
nan_count = (full_data['clean_description'].astype(str) == 'nan').sum()
print(f"\nNombre de description NaN : {nan_count}")


Nombre de description NaN : 0


In [99]:
display(full_data.head(20))
test_data = full_data.drop(columns=['designation', 'description', 'productid', 'imageid','clean_designation','clean_description'], axis=1)
display(test_data.head(20))

,designation,description,productid,imageid,prdtypecode,clean_designation,clean_description,text
id,,,,,,,,
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,10,olivia: personalisiertes notizbuch / 150 seite...,,olivia: personalisiertes notizbuch / 150 seite...
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,2280,journal des arts (le) n° 133 du 28/09/2001 - l...,,journal des arts (le) n° 133 du 28/09/2001 - l...
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,grand stylet ergonomique bleu gamepad nintendo...,pilot style touch pen de marque speedlink est ...,grand stylet ergonomique bleu gamepad nintendo...
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,1280,peluche donald - europe - disneyland 2000 (mar...,,peluche donald - europe - disneyland 2000 (mar...
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,la guerre des tuques,luc a des ide;es de grandeur. il veut organise...,la guerre des tuques luc a des ide;es de grand...
5,Afrique Contemporaine N° 212 Hiver 2004 - Doss...,NaN,5862738,393356830,2280,afrique contemporaine n° 212 hiver 2004 - doss...,,afrique contemporaine n° 212 hiver 2004 - doss...
6,Christof E: Bildungsprozessen Auf Der Spur,NaN,91920807,907794536,10,christof e: bildungsprozessen auf der spur,,christof e: bildungsprozessen auf der spur
7,Conquérant Sept Cahier Couverture Polypro 240 ...,CONQUERANT CLASSIQUE Cahier 240 x 320 mm seyès...,344240059,999581347,2522,conquérant sept cahier couverture polypro 240 ...,conquerant classique cahier 240 x 320 mm seyès...,conquérant sept cahier couverture polypro 240 ...
8,Puzzle Scooby-Doo Avec Poster 2x35 Pieces,NaN,4239126071,1325918866,1280,puzzle scooby-doo avec poster 2x35 pieces,,puzzle scooby-doo avec poster 2x35 pieces


,prdtypecode,text
id,,
0,10,olivia: personalisiertes notizbuch / 150 seite...
1,2280,journal des arts (le) n° 133 du 28/09/2001 - l...
2,50,grand stylet ergonomique bleu gamepad nintendo...
3,1280,peluche donald - europe - disneyland 2000 (mar...
4,2705,la guerre des tuques luc a des ide;es de grand...
5,2280,afrique contemporaine n° 212 hiver 2004 - doss...
6,10,christof e: bildungsprozessen auf der spur
7,2522,conquérant sept cahier couverture polypro 240 ...
8,1280,puzzle scooby-doo avec poster 2x35 pieces


In [ ]:
# Analyse des langues du jeu de données
# from langdetect import detect, DetectorFactory, LangDetectException
# USE_LANG_DETECT = True  # Mettre à True pour activer la détection de langue
# DetectorFactory.seed = 0  # Pour des résultats reproductibles

# def detect_language(text):
#    if not USE_LANG_DETECT:
#        return 'unknown'
#    try:
#        return detect(text)
#    except (LangDetectException, Exception):
#        return 'unknown'
    

#test_data['language'] = test_data['text'].apply(detect_language)
#print(test_data['language'].value_counts())

KeyboardInterrupt: 